# 🚀 Fine-Tuning Career PathFinder AI Agent (Llama 3.2 3B)
This notebook fine-tunes **Llama-3.2-3B-Instruct** using **Unsloth (QLoRA 4-bit)** on a free Google Colab T4 GPU in ~15 minutes.

### Agent Capabilities Trained:
1. **Career Research Agent**: Role extraction, core skills, emerging tech, and salary/demand trends.
2. **Skill Gap Agent**: Proficiency scoring and gap prioritization (High/Medium/Low).
3. **Roadmap Agent**: Phased learning path synthesis with milestone projects.
4. **Career Advisor Agent**: Evidence-based readiness scores with `[FACT]` / `[INFERENCE]` / `[RECOMMENDATION]` tags.
5. **Interview & Assessment Agent**: Technical questions, grading rubrics, and conceptual MCQs.

## 1. Install Unsloth & Dependencies
Run this on a GPU runtime (Runtime > Change runtime type > T4 GPU).

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets transformers trl

## 2. Load Base Model & Tokenizer (Llama 3.2 3B Instruct 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 3. Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Load Training & Validation Dataset

In [ ]:
from datasets import load_dataset

# Upload your train.jsonl and val.jsonl or load from URL/disk
# If uploading to Colab files panel:
dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset['train'])} training examples, {len(dataset['validation'])} validation examples.")

## 5. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="outputs",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

trainer_stats = trainer.train()

## 6. Test Model Inference

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are the Career Advisor AI Agent for Career PathFinder. Your task is to deliver calibrated, evidence-backed career readiness evaluations. RULES: 1. Never say 'You will definitely become X' - always say 'Current readiness: Y%'. 2. State your confidence level (high, moderate, low, insufficient) based on available evidence. 3. Structure your reasoning with clear [FACT], [INFERENCE], and [RECOMMENDATION] tags. Output strictly valid JSON."},
    {"role": "user", "content": "Generate evidence-based career advice for user targeting 'AI Engineer'. Profile: Experience Level 'Junior Professional', Assessments Completed: 3, Quizzes Completed: 8 (Average score: 88%), Projects Completed: 1."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## 7. Export GGUF for Ollama (Local Deployment)

In [ ]:
# Export directly to GGUF Q4_K_M
model.save_pretrained_gguf("career_agent_gguf", tokenizer, quantization_method="q4_k_m")

# Download the GGUF file:
from google.colab import files
!zip -r career_agent_gguf.zip career_agent_gguf
files.download("career_agent_gguf.zip")